# 📚 WeebCentral Manga Downloader

Download manga chapters from WeebCentral with full parallel support.

### Setup
1. Run **Cell 1** once to install dependencies + Clone Repository.
2. Run **Cell 2** — paste URL, pick chapters & format, everything downloads.
3. Optionally run **Cell 3** to zip & download to your PC.

### Output formats
| Option | Result |
|--------|--------|
| `pdf` | One PDF per chapter |
| `cbz` | CBZ archive with `ComicInfo.xml` (Kavita / Komga / CDisplayEx) |
| `images` | Raw image folder per chapter |
| `all` | PDF + CBZ + Images |

### Supported URL
```
https://weebcentral.com/series/SERIES_ID/manga-slug
```

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 1 — Install dependencies  (run once)          ║
# ╚══════════════════════════════════════════════════════╝
!pip install -q requests 'httpx[http2]' nest_asyncio beautifulsoup4 lxml Pillow fpdf2 tqdm
!pip install PyPDF2
print('✅ Dependencies ready.')
!git clone https://github.com/Yui007/weebcentral_downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00
✅ Dependencies ready.
fatal: destination path 'weebcentral_downloader' already exists and is not an empty directory.


In [4]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 2 — Scrape, select & download                 ║
# ╚══════════════════════════════════════════════════════╝
import sys
import os
from PyPDF2 import PdfMerger
from pathlib import Path

sys.path.insert(0, '/content/weebcentral_downloader/colab')

from colab_scraper    import scrape_manga_info, scrape_chapter_list
from colab_downloader import parse_chapter_selection, download_chapters

def merge_pdfs_to_target_size(pdf_files, output_path, target_size_mb=200):
    """
    Merge PDF files into larger files up to target_size_mb.
    Returns list of created merged files.
    """
    target_bytes = target_size_mb * 1024 * 1024
    merged_files = []

    if not pdf_files:
        return merged_files

    current_batch = []
    current_size = 0

    for pdf_file in pdf_files:
        file_size = os.path.getsize(pdf_file)

        # If this file alone exceeds target size, create separate file for it
        if file_size > target_bytes:
            # Save current batch if exists
            if current_batch:
                merged_file = merge_pdf_batch(current_batch, output_path, len(merged_files) + 1)
                merged_files.append(merged_file)
                current_batch = []
                current_size = 0

            # Create single file for this large PDF
            single_merged = merge_pdf_batch([pdf_file], output_path, len(merged_files) + 1)
            merged_files.append(single_merged)
            continue

        # Check if adding this file would exceed target size
        if current_size + file_size > target_bytes and current_batch:
            # Save current batch and start new one
            merged_file = merge_pdf_batch(current_batch, output_path, len(merged_files) + 1)
            merged_files.append(merged_file)
            current_batch = [pdf_file]
            current_size = file_size
        else:
            # Add to current batch
            current_batch.append(pdf_file)
            current_size += file_size

    # Save the last batch
    if current_batch:
        merged_file = merge_pdf_batch(current_batch, output_path, len(merged_files) + 1)
        merged_files.append(merged_file)

    return merged_files

def merge_pdf_batch(pdf_list, output_dir, batch_num):
    """Merge a list of PDF files into one."""
    if not pdf_list:
        return None

    merger = PdfMerger()
    for pdf_file in pdf_list:
        merger.append(pdf_file)

    # Create output filename with batch number
    base_name = Path(pdf_list[0]).stem.split('_')[0]  # Get manga name from first file
    output_file = os.path.join(output_dir, f"{base_name}_batch_{batch_num}.pdf")

    # Ensure we don't overwrite existing files
    counter = 1
    while os.path.exists(output_file):
        output_file = os.path.join(output_dir, f"{base_name}_batch_{batch_num}_{counter}.pdf")
        counter += 1

    merger.write(output_file)
    merger.close()

    return output_file

def process_merged_pdfs(original_dir, merged_dir, target_size_mb=200):
    """Process all PDFs in original_dir and create merged versions in merged_dir."""
    # Create merged directory if it doesn't exist
    os.makedirs(merged_dir, exist_ok=True)

    # Find all PDF files in original directory
    pdf_files = sorted([os.path.join(original_dir, f) for f in os.listdir(original_dir)
                        if f.endswith('.pdf') and not f.startswith('merged_')])

    if not pdf_files:
        return []

    # Merge PDFs
    merged_files = merge_pdfs_to_target_size(pdf_files, merged_dir, target_size_mb)

    return merged_files

# ── 1. URL ───────────────────────────────────────────────────────────────────
SERIES_URL = input(
    '🌐 WeebCentral series URL\n'
    '   e.g. https://weebcentral.com/series/01JCZQE.../manga-slug\n'
    '   > '
).strip()

# ── 2. Scrape ────────────────────────────────────────────────────────────────
manga_info = scrape_manga_info(SERIES_URL)
chapters   = scrape_chapter_list(SERIES_URL)
total      = len(chapters)

# ── 3. Show chapter list ─────────────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  {'#':>4}   {'Chapter':<50}  Date")
print(f"{'='*62}")
for ch in chapters:
    num  = str(ch['index']).rjust(4)
    name = ch['title'][:50].ljust(50)
    print(f"  {num}   {name}  {ch['date']}")
print(f"{'='*62}")
print(f"  Total: {total} chapters\n")

# ── 4. Chapter selection ─────────────────────────────────────────────────────
print('📥 Selection formats:')
print('   all          → every chapter')
print('   single 5     → chapter 5 only')
print('   range 1-10   → chapters 1 through 10 (inclusive)')
print('   1,5,9,15     → specific chapters')
print()

while True:
    sel_input = input(f'🎯 Select chapters (1–{total}): ').strip()
    try:
        selected = parse_chapter_selection(sel_input, total)
        print(f'   ✅ {len(selected)} chapter(s) selected.')
        break
    except ValueError as e:
        print(f'   ❌ {e}\n   Try again.')

# ── 5. Output format ─────────────────────────────────────────────────────────
print()
print('📦 Output format:')
print('   pdf    → PDF file per chapter')
print('   cbz    → CBZ with ComicInfo.xml  (Kavita / Komga / CDisplayEx)')
print('   images → Raw image folder per chapter')
print('   all    → PDF + CBZ + Images')
print()

while True:
    fmt = input('🗂  Format [pdf / cbz / images / all]: ').strip().lower()
    if fmt in ('pdf', 'cbz', 'images', 'all'):
        break
    print('   ❌ Please enter pdf, cbz, images, or all.')

# ── 6. Download ──────────────────────────────────────────────────────────────
output_dir = download_chapters(
    manga_info       = manga_info,
    chapters         = chapters,
    selected_indices = selected,
    output_format    = fmt,
    output_dir       = '/content/weebcentral_downloader/colab/manga',
)

# ── 7. Merge PDFs if PDF format was requested ────────────────────────────────
if fmt in ('pdf', 'all'):
    print("\n🔄 Merging PDFs to target size (max 200MB per file)...")

    # Define directories
    original_pdf_dir = output_dir  # PDFs are saved in the main output directory
    merged_pdf_dir = os.path.join(output_dir, 'merged_pdfs')

    # Process and merge PDFs
    merged_files = process_merged_pdfs(original_pdf_dir, merged_pdf_dir, target_size_mb=200)

    if merged_files:
        print(f"✅ Created {len(merged_files)} merged PDF file(s) in '{merged_pdf_dir}':")
        for i, mf in enumerate(merged_files, 1):
            size_mb = os.path.getsize(mf) / (1024 * 1024)
            print(f"   {i}. {os.path.basename(mf)} ({size_mb:.2f} MB)")

        print(f"\n📁 Original PDFs (per chapter) are still available in: {original_pdf_dir}")
        print(f"📁 Merged PDFs are available in: {merged_pdf_dir}")
    else:
        print("⚠️  No PDF files found to merge.")

🌐 WeebCentral series URL
   e.g. https://weebcentral.com/series/01JCZQE.../manga-slug
   > https://weebcentral.com/series/01J76XYA2AFH8MNBG4FRCM5JMV/Oyasumi-Punpun
🔎 Fetching series page: https://weebcentral.com/series/01J76XYA2AFH8MNBG4FRCM5JMV/Oyasumi-Punpun

📖 Title    : Goodnight Punpun
👤 Authors  : ASANO Inio
🏷  Tags     : Adult, Comedy, Drama, Psychological, Seinen ...
📌 Type     : Manga
📊 Status   : Complete
📅 Released : 2007
🖼  Cover    : https://temp.compsci88.com/cover/fallback/01J76XYA2AFH8MNBG4FRCM5JMV.jpg

📝 Description:
Meet Punpun Punyama. He’s an average kid in an average town. He wants to win a Nobel Prize and save the world.
He wants the girl he has a crush on to like him back. He wants to find some porn. That’s what he wants, but what does he get?

🔎 Fetching chapter list: https://weebcentral.com/series/01J76XYA2AFH8MNBG4FRCM5JMV/full-chapter-list

📚 Total chapters: 147
   Ch 1 (oldest) : Chapter 1 Last Read 2024-09-07T17:04:15.717343Z
   Ch 147 (latest) : Chapter 14

  Ch001:   0%|          | 0/31 [00:00<?, ?pg/s]

       🖼  23 pages — downloading in parallel...


  Ch003:   0%|          | 0/23 [00:00<?, ?pg/s]

       🖼  18 pages — downloading in parallel...


...

          → Goodnight Punpun - Ch143 - Chapter 143 Last Read 2024-09-07T17_04_15.717343Z.pdf

[146/147] 📖  Chapter 146 Last Read 2024-09-07T17:04:15.717343Z  (2024-09-07T17:04:15.717343Z)
       ✅ 22/22 pages
       📄 Building PDF...
       🖼  18 pages — downloading in parallel...


  Ch146:   0%|          | 0/18 [00:00<?, ?pg/s]

       ✅ 20/20 pages
       📄 Building PDF...
          → Goodnight Punpun - Ch144 - Chapter 144 Last Read 2024-09-07T17_04_15.717343Z.pdf

[147/147] 📖  Chapter 147 Last Read 2024-09-07T17:04:15.717343Z  (2024-09-07T17:04:15.717343Z)
       ✅ 18/18 pages
       📄 Building PDF...
       🖼  29 pages — downloading in parallel...


  Ch147:   0%|          | 0/29 [00:00<?, ?pg/s]

          → Goodnight Punpun - Ch145 - Chapter 145 Last Read 2024-09-07T17_04_15.717343Z.pdf

          → Goodnight Punpun - Ch146 - Chapter 146 Last Read 2024-09-07T17_04_15.717343Z.pdf

       ✅ 29/29 pages
       📄 Building PDF...
          → Goodnight Punpun - Ch147 - Chapter 147 Last Read 2024-09-07T17_04_15.717343Z.pdf

───────────────────────────────────────────────────────
🎉 Finished in 1m 35s  —  saved to:
   /content/weebcentral_downloader/colab/manga/Goodnight Punpun

🔄 Merging PDFs to target size (max 200MB per file)...
✅ Created 9 merged PDF file(s) in '/content/weebcentral_downloader/colab/manga/Goodnight Punpun/merged_pdfs':
   1. Goodnight Punpun - Ch001 - Chapter 1 Last Read 2024-09-07T17_batch_1_1.pdf (190.73 MB)
   2. Goodnight Punpun - Ch022 - Chapter 22 Last Read 2024-09-07T17_batch_2_1.pdf (194.58 MB)
   3. Goodnight Punpun - Ch041 - Chapter 41 Last Read 2024-09-07T17_batch_3.pdf (199.50 MB)
   4. Goodnight Punpun - Ch059 - Chapter 59 Last Read 2024-09-07T17_batch

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — (Optional) Zip & download to your PC      ║
# ╚══════════════════════════════════════════════════════╝
import re
import shutil
from google.colab import files
import os

try:
    # Создаем безопасное имя для архива
    zip_stem = re.sub(r'[\\/:*?"<>|]', '_', manga_info['title']).strip() + '_manga'
    zip_path = f'/content/weebcentral_downloader/colab/{zip_stem}'

    print(f'📦 Creating archive from: {output_dir}')

    # Проверяем, существует ли директория
    if not os.path.exists(output_dir):
        print(f'❌ Error: Directory {output_dir} does not exist!')
    else:
        # Создаем zip архив
        shutil.make_archive(zip_path, 'zip', output_dir)

        # Полный путь к созданному архиву
        zip_file = f'{zip_path}.zip'

        if os.path.exists(zip_file):
            # Получаем размер архива
            size_mb = os.path.getsize(zip_file) / (1024 * 1024)
            print(f'✅ Archive created: {zip_stem}.zip ({size_mb:.2f} MB)')
            print(f'📥 Initiating download...')

            # Скачиваем архив
            files.download(zip_file)

            print('✅ Download completed!')
        else:
            print(f'❌ Archive file not found: {zip_file}')

except Exception as e:
    print(f'❌ Error during archiving/download: {str(e)}')

📦 Creating archive from: /content/weebcentral_downloader/colab/manga/Goodnight Punpun
